# Sprint 4: RAG Prompt, Answer, and Evaluation

This notebook demonstrates the full Sprint 4 backend flow for the Mini Wikipedia RAG system.

## What this notebook covers
1. Confirm the API server is running
2. Initialize the vector index
3. Construct an augmented prompt from retrieved documents
4. Generate an answer with GPT-4o
5. Evaluate answer quality against reference answers

**Prerequisites**: start the server in a separate terminal before running these cells.
```
uv run uvicorn api:app --app-dir src --host 127.0.0.1 --port 8000
```

In [1]:
import sys
sys.path.insert(0, '../src')

import json
import requests

BASE_URL = 'http://127.0.0.1:8000'
QUERY_TEXT = 'What is Python?'
TOP_K = 3

print(f'Using server at {BASE_URL}')
print(f'Test query: {QUERY_TEXT}')

Using server at http://127.0.0.1:8000
Test query: What is Python?


## 1. Check Server Readiness

Confirm that the FastAPI server is reachable and has been restarted with the Sprint 4 routes.

In [2]:
try:
    response = requests.get(f'{BASE_URL}/openapi.json')
    print(f'Status code: {response.status_code}')

    if response.status_code == 200:
        routes = response.json().get('paths', {})
        print('\nSprint 4 routes present:')
        for route in ['/rag/prompt', '/rag/answer', '/rag/evaluate']:
            print(f'  {route}: {route in routes}')
    else:
        print(f'Error {response.status_code}: {response.text[:200]}')

except requests.exceptions.ConnectionError:
    print('❌ Connection error: Is the server running?')
    print('   Start with: uv run uvicorn api:app --app-dir src --host 127.0.0.1 --port 8000 --reload')
except Exception as e:
    print(f'❌ Unexpected error: {type(e).__name__}: {str(e)}')

Status code: 200

Sprint 4 routes present:
  /rag/prompt: True
  /rag/answer: True
  /rag/evaluate: True


## 2. Initialize the Vector Index

Load the sample documents and create the vector index used by the RAG endpoints.

In [3]:
try:
    # Load the full HuggingFace corpus: 3,200 passages total
    # The 918 test questions are drawn from these passages, so loading all of them
    # gives the best possible retrieval accuracy and evaluation scores.
    response = requests.post(f'{BASE_URL}/ingest', params={'document_count': 3200, 'use_sample': False})
    print(f'Status code: {response.status_code}')

    if response.status_code == 200:
        ingest_result = response.json()
        print(json.dumps(ingest_result, indent=2))
        print(f'\n✓ Ingested {ingest_result["documents_ingested"]} / 3200 documents from {ingest_result.get("source", "unknown")}')
        print(f'  Full corpus loaded → best retrieval coverage for the 918 test questions')
    else:
        print(f'Error {response.status_code}: {response.text[:300]}')
        print('\n⚠️ If ingestion fails here, check Azure auth: azd auth login --scope api://ailab/Model.Access')

except requests.exceptions.ConnectionError:
    print('❌ Connection error: Is the server running?')
except Exception as e:
    print(f'❌ Unexpected error: {type(e).__name__}: {str(e)}')

Status code: 200
{
  "status": "success",
  "documents_ingested": 3200,
  "index_ready": true,
  "source": "huggingface"
}

✓ Ingested 3200 / 3200 documents from huggingface
  Full corpus loaded → best retrieval coverage for the 918 test questions


## 3. Construct the Augmented Prompt

Inspect the prompt that combines the user query with the retrieved documents.

In [4]:
prompt_payload = {'query': QUERY_TEXT, 'top_k': TOP_K}
prompt_result = None

try:
    response = requests.post(f'{BASE_URL}/rag/prompt', json=prompt_payload)
    print(f'Status code: {response.status_code}')

    if response.status_code == 200:
        prompt_result = response.json()
        print(f'Embedding dimension: {prompt_result["query_embedding_dimension"]}')
        print(f'Retrieved documents: {len(prompt_result["documents"])}')
        print('\nPrompt preview:\n')
        print(prompt_result['prompt'][:1200])
    elif response.status_code == 404:
        print('❌ The Sprint 4 endpoints are not available on the running server.')
        print('   Restart with: uv run uvicorn api:app --app-dir src --host 127.0.0.1 --port 8000 --reload')
    else:
        print(f'Error {response.status_code}: {response.text[:300]}')

except requests.exceptions.ConnectionError:
    print('❌ Connection error: Is the server running?')
except Exception as e:
    print(f'❌ Unexpected error: {type(e).__name__}: {str(e)}')

Status code: 200
Embedding dimension: 3072
Retrieved documents: 3

Prompt preview:

You are a retrieval-augmented assistant. Answer the user question using only the provided context. If the context is insufficient, say that directly.

User Question:
What is Python?

Retrieved Context:
[1] Document 1 score=0.158
Turtles, particularly small terrestrial and freshwater turtles, are commonly kept as pets. Among the most popular are Russian Tortoises, Greek spur-thighed tortoises and red-ear sliders (or terrapin). David Alderton (1986). An Interpet Guide to Reptiles & Amphibians, Salamander Books Ltd., London & New York.

[2] Document 2 score=0.154
Tux the Linux kernel mascot

[3] Document 3 score=0.152
The generic component of its modern scientific designation, Panthera pardus, is derived from Latin via Greek ÏÎ¬Î½Î¸Î·Ï pÃ¡nthÄr. A folk etymology held that it was a compound of ÏÎ±Î½ pan ("all") and Î¸Î·Ï ("beast"). However, it is believed instead to derive from an Indo-Iranian word mea

## 4. Generate an Answer

Call GPT-4o through the Sprint 4 answer endpoint and inspect the grounded answer returned by the backend.

In [5]:
answer_payload = {'query': QUERY_TEXT, 'top_k': TOP_K}
answer_result = None

try:
    response = requests.post(f'{BASE_URL}/rag/answer', json=answer_payload)
    print(f'Status code: {response.status_code}')

    if response.status_code == 200:
        answer_result = response.json()
        print('Generated answer:\n')
        print(answer_result['answer'])
        print(f'\nRetrieved documents: {len(answer_result["documents"])}')
    else:
        print(f'Error {response.status_code}: {response.text[:300]}')

except requests.exceptions.ConnectionError:
    print('❌ Connection error: Is the server running?')
except Exception as e:
    print(f'❌ Unexpected error: {type(e).__name__}: {str(e)}')

Status code: 200
Generated answer:

The provided context does not contain information about Python.

Retrieved documents: 3


## 5. Evaluate the RAG Pipeline

Run the lightweight evaluation endpoint to compare generated answers against reference answers.

In [6]:
try:
    response = requests.get(f'{BASE_URL}/rag/evaluate', params={'limit': 3, 'top_k': TOP_K})
    print(f'Status code: {response.status_code}')

    if response.status_code == 200:
        evaluation_result = response.json()
        print(json.dumps(evaluation_result, indent=2))
        print(f'\nAverage score: {evaluation_result["average_score"]}')
        print(f'Passed examples: {evaluation_result["passed_count"]}/{evaluation_result["example_count"]}')
    else:
        print(f'Error {response.status_code}: {response.text[:300]}')

except requests.exceptions.ConnectionError:
    print('❌ Connection error: Is the server running?')
except Exception as e:
    print(f'❌ Unexpected error: {type(e).__name__}: {str(e)}')

Status code: 200
{
  "example_count": 3,
  "average_score": 0.157,
  "passed_count": 0,
  "results": [
    {
      "query": "Was Abraham Lincoln the sixteenth President of the United States?",
      "expected_answer": "yes",
      "generated_answer": "Yes, Abraham Lincoln was the sixteenth President of the United States.",
      "score": 0.205
    },
    {
      "query": "Did Lincoln sign the National Banking Act of 1863?",
      "expected_answer": "yes",
      "generated_answer": "Yes, Lincoln signed the National Banking Act of 1863, which was part of the legislation that created a system of national banks and allowed for a strong national financial system.",
      "score": 0.147
    },
    {
      "query": "Did his mother die of pneumonia?",
      "expected_answer": "no",
      "generated_answer": "No, his mother did not die of pneumonia. According to the context, she died of milk sickness.",
      "score": 0.118
    }
  ]
}

Average score: 0.157
Passed examples: 0/3


## Sprint 4 Outcome Checklist

If the cells above succeed, Sprint 4 outcomes are covered:
- `/rag/prompt` constructs an enhanced prompt from retrieved documents
- `/rag/answer` sends the prompt to GPT-4o and returns a response
- `/rag/evaluate` compares generated answers against reference answers
- the backend now exposes the complete retrieval-plus-answer path

## 6. Comprehensive Evaluation with HuggingFace Test Dataset

Run evaluation across multiple test questions from the HuggingFace rag-mini-wikipedia test set to measure RAG system performance.

In [ ]:
import pandas as pd
import numpy as np

# ── Evaluation parameters ────────────────────────────────────────────────────
# Each example = 1 retrieve + 1 LLM call + 2 embedding calls for scoring.
# Cost per example: ~5–15s. Adjust EVAL_LIMIT to control total runtime:
#   5  examples ≈ 1–2 min
#   20 examples ≈ 5–10 min
#   All 918     ≈ several hours (not recommended interactively)
#
# Scoring mode (backend):
# - yes/no references: explicit polarity check
# - general answers: 0.7 * semantic cosine + 0.3 * lexical overlap
EVAL_LIMIT = 5
EVAL_TOP_K = 3
PASS_THRESHOLD = 0.3
# ─────────────────────────────────────────────────────────────────────────────

eval_attempts = []
eval_scores = []
eval_errors = []

print(f"Running evaluation on {EVAL_LIMIT} test examples (top_k={EVAL_TOP_K})...")
print("Scoring method: hybrid (semantic + lexical), with yes/no handling.\n")

try:
    response = requests.get(f'{BASE_URL}/rag/evaluate', params={'limit': EVAL_LIMIT, 'top_k': EVAL_TOP_K})
    print(f'Status code: {response.status_code}')

    if response.status_code == 200:
        evaluation_result = response.json()

        results = evaluation_result.get('results', [])
        example_count = evaluation_result.get('example_count', 0)
        average_score = evaluation_result.get('average_score', 0)
        passed_count = evaluation_result.get('passed_count', 0)

        print(f'\n📊 Evaluation Summary:')
        print(f'   Examples tested          : {example_count}')
        print(f'   Average hybrid score     : {average_score:.4f}')
        print(f'   Passed (score ≥ {PASS_THRESHOLD:.1f})     : {passed_count}/{example_count}')
        if example_count > 0:
            print(f'   Pass rate                : {passed_count/example_count*100:.1f}%')

        eval_attempts = results
        eval_scores = [r.get('score', 0) for r in results]

        print(f'\n📈 Score distribution:')
        if eval_scores:
            print(f'   Min    : {min(eval_scores):.4f}')
            print(f'   Max    : {max(eval_scores):.4f}')
            print(f'   Median : {np.median(eval_scores):.4f}')
            print(f'   Std dev: {np.std(eval_scores):.4f}')

        print('\n✅ Results stored — run the analysis cells below.')
    else:
        print(f'Error {response.status_code}: {response.text[:300]}')

except requests.exceptions.ConnectionError:
    print('❌ Connection error: Is the server running?')
except Exception as e:
    print(f'❌ {type(e).__name__}: {e}')

Running evaluation on 5 test examples (top_k=3)...
Scoring method: hybrid (semantic + lexical), with yes/no handling.



## 7. Detailed Example Analysis

Examine individual test cases to understand which queries work well and which need improvement.

In [11]:
if eval_attempts:
    # Sort by score to find best and worst examples
    sorted_examples = sorted(eval_attempts, key=lambda x: x.get('score', 0))
    
    print("🎯 Best Performing Examples (Highest Scores):\n")
    best_examples = sorted(eval_attempts, key=lambda x: x.get('score', 0), reverse=True)[:3]
    for i, example in enumerate(best_examples, 1):
        score = example.get('score', 0)
        query = example.get('query', '')
        print(f"{i}. Score: {score:.4f} | Query: {query[:70]}...")
    
    print(f"\n⚠️  Challenging Examples (Lowest Scores):\n")
    worst_examples = sorted_examples[:3]
    for i, example in enumerate(worst_examples, 1):
        score = example.get('score', 0)
        query = example.get('query', '')
        print(f"{i}. Score: {score:.4f} | Query: {query[:70]}...")
    
    print(f"\n📋 Detailed View of First Example:\n")
    if eval_attempts:
        first = eval_attempts[0]
        print(f"Question: {first.get('query', 'N/A')}")
        print(f"\nExpected Answer:\n{first.get('expected_answer', 'N/A')[:300]}...")
        print(f"\nGenerated Answer:\n{first.get('generated_answer', 'N/A')[:300]}...")
        print(f"\nSimilarity Score: {first.get('score', 0):.4f}")
else:
    print("No evaluation results available. Run the evaluation cell above first.")

🎯 Best Performing Examples (Highest Scores):

1. Score: 0.3120 | Query: How many long was Lincoln's formal education?...
2. Score: 0.2970 | Query: When did Lincoln begin his political career?...
3. Score: 0.2050 | Query: Was Abraham Lincoln the sixteenth President of the United States?...

⚠️  Challenging Examples (Lowest Scores):

1. Score: 0.1150 | Query: Did his mother die of pneumonia?...
2. Score: 0.1470 | Query: Did Lincoln sign the National Banking Act of 1863?...
3. Score: 0.2050 | Query: Was Abraham Lincoln the sixteenth President of the United States?...

📋 Detailed View of First Example:

Question: Was Abraham Lincoln the sixteenth President of the United States?

Expected Answer:
yes...

Generated Answer:
Yes, Abraham Lincoln was the sixteenth President of the United States....

Similarity Score: 0.2050


## 8. Performance Analysis and Insights

Analyze evaluation metrics to identify patterns and improvement opportunities.

In [ ]:
# Analysis and insights
if eval_scores:
    df_results = pd.DataFrame(eval_attempts)

    pass_threshold = 0.3

    print('📊 Performance Metrics Summary\n')
    print(f'{"Metric":<30} {"Value":>10}')
    print('-' * 42)
    print(f'{"Total Examples":<30} {len(eval_scores):>10}')
    print(f'{"Average Score":<30} {np.mean(eval_scores):>10.4f}')
    print(f'{"Median Score":<30} {np.median(eval_scores):>10.4f}')
    print(f'{"Std Deviation":<30} {np.std(eval_scores):>10.4f}')
    print(f'{"Min Score":<30} {np.min(eval_scores):>10.4f}')
    print(f'{"Max Score":<30} {np.max(eval_scores):>10.4f}')
    print(f'{"25th Percentile":<30} {np.percentile(eval_scores, 25):>10.4f}')
    print(f'{"75th Percentile":<30} {np.percentile(eval_scores, 75):>10.4f}')

    # Score threshold analysis
    thresholds = [0.2, 0.3, 0.5, 0.7]
    print(f'\n📈 Score Distribution by Threshold\n')
    print(f'{"Threshold":<15} {"Pass Count":<12} {"Pass Rate":>10}')
    print('-' * 37)
    for threshold in thresholds:
        pass_count = sum(1 for s in eval_scores if s >= threshold)
        pass_rate = (pass_count / len(eval_scores) * 100) if eval_scores else 0
        print(f'{threshold:<15.1f} {pass_count:<12} {pass_rate:>9.1f}%')

    print(f'\n💡 Key Insights:\n')
    print(f'✓ The RAG system achieves an average hybrid score of {np.mean(eval_scores):.4f}')
    print(f'✓ {sum(1 for s in eval_scores if s >= pass_threshold)} out of {len(eval_scores)} examples ({sum(1 for s in eval_scores if s >= pass_threshold)/len(eval_scores)*100:.1f}%) pass the {pass_threshold:.1f} threshold')

    if np.std(eval_scores) > 0.15:
        print(f'⚠ High variability in scores (std: {np.std(eval_scores):.4f}) suggests retrieval quality inconsistency')
    else:
        print(f'✓ Consistent performance across examples (std: {np.std(eval_scores):.4f})')

    print(f'\n🔧 Recommendations:\n')
    if np.mean(eval_scores) < pass_threshold:
        print('   • Increase document coverage and ensure ingest completes successfully')
        print('   • Review top_k and retrieval relevance for weak examples')
        print('   • Inspect prompt grounding for yes/no and short-answer questions')
    else:
        print('   • Current configuration shows acceptable baseline performance')
        print('   • Focus on edge cases with low scores')
        print('   • Consider evaluating on larger subsets (20, 50, 100)')
else:
    print("No evaluation results available to analyze.")

📊 Performance Metrics Summary

Metric                              Value
------------------------------------------
Total Examples                          5
Average Score                      0.2152
Median Score                       0.2050
Std Deviation                      0.0786
Min Score                          0.1150
Max Score                          0.3120
25th Percentile                    0.1470
75th Percentile                    0.2970

📈 Score Distribution by Threshold

Threshold       Pass Count    Pass Rate
-------------------------------------
0.3             1                 20.0%
0.5             0                  0.0%
0.7             0                  0.0%
0.9             0                  0.0%

💡 Key Insights:

✓ The RAG system achieves an average semantic similarity of 0.2152
✓ 0 out of 5 examples (0.0%) pass the 0.5 threshold
✓ Consistent performance across examples (std: 0.0786)

🔧 Recommendations:

   • Consider increasing document count during ingestion
   •

## 9. RAG Pipeline Flow Summary

The complete RAG pipeline demonstrates:

1. **Query Augmentation**: User query + retrieved documents → augmented prompt
2. **Answer Generation**: Augmented prompt → GPT-4o LLM → synthesized answer
3. **Evaluation**: Compare generated answers against reference answers using semantic similarity

### Key Components

- **Retrieval**: Vector similarity search finds top-K relevant documents from the knowledge base
- **Augmentation**: Retrieved context is combined with the original query to create a rich prompt
- **Generation**: GPT-4o uses the augmented prompt to generate a grounded, contextual answer
- **Evaluation**: Semantic similarity scoring measures how closely generated answers match reference answers

### Test Dataset

The evaluation uses the HuggingFace `rag-mini-wikipedia` test set:
- Test questions: `hf://datasets/rag-datasets/rag-mini-wikipedia/data/test.parquet/part.0.parquet`
- Reference answers: Pre-labeled answers for each test question
- Coverage: 918 total test examples available for evaluation

### Next Steps

To improve RAG system performance:
1. Increase document count in the knowledge base for better retrieval coverage
2. Experiment with different retrieval strategies (e.g., BM25, hybrid search)
3. Fine-tune prompts for better answer generation
4. Evaluate on larger subsets of the test dataset
5. Implement caching and optimization for production deployment